# RunPod Ollama 외부 API 접속

RunPod GPU Pod 안에서 실행 중인 Ollama HTTP API를 로컬 PyCharm에서 호출한다. 이 과정은 외부 접속 구조를 이해하기 위한 실습이며 인증·요청 제한·감사 로그를 갖춘 운영 배포와는 구분한다.

`PyCharm → RunPod HTTPS proxy → Pod의 11434 port → Ollama → GPU model → 응답`

01번 노트북에서 Ollama 설치와 `llama3.2` 다운로드를 마친 상태를 전제로 한다. 이 노트북은 기본 모델을 RunPod 외부에 공개하고 native API와 Python client로 호출하는 과정에 집중한다. 새 Pod로 시작한다면 아래 RunPod 설정부터 순서대로 진행한다. 설치 기준은 [RunPod Ollama 공식 가이드](https://docs.runpod.io/tutorials/pods/run-ollama)를 따른다.

## 실행 위치 구분

실행 위치를 다음과 같이 구분한다.

1. **RunPod Web Terminal**: `localhost`로 server와 model을 먼저 확인한다.
2. **로컬 PyCharm Terminal**: RunPod HTTPS proxy로 외부 접속을 확인한다.
3. **로컬 Python kernel**: `Client(host=...)`와 `ChatOllama`로 같은 server를 호출한다.

PyCharm에 연결한 RunPod 외부 Jupyter kernel에서 Python 셀을 실행하면 요청도 Pod 내부에서 발생하므로 외부 PC 접속을 검증한 것이 아니다. 이 노트북의 Python 셀은 로컬 interpreter를 선택한다.

> **보안 주의:** RunPod HTTP proxy로 공개한 Ollama API에는 기본 인증이 없다. 실습 중 짧게 연결을 확인할 때만 사용하고 민감한 데이터를 보내지 않는다. 실습 후 Pod를 중지하거나 삭제하며, 실제 서비스에는 인증 proxy와 rate limit을 추가한다.

## RunPod Pod 생성과 필수 설정

RunPod의 HTTP proxy로 Ollama API를 호출하려면 Pod를 생성할 때 내부 port와 server binding을 함께 설정해야 한다. 공식 문서의 A40은 예시 GPU이며 필수 사양이 아니다. 수업 기본 모델인 `llama3.2`와 생성 중 사용하는 KV cache가 들어갈 만큼 VRAM이 있는 GPU를 선택하고, 더 큰 모델이나 긴 context를 사용할 때는 모델 크기에 맞춰 GPU를 올린다.

1. RunPod console의 **Pods**에서 **Deploy**를 선택한다.
2. 사용할 GPU와 최신 **PyTorch template**을 선택한다.
3. **Pod Template → Edit**를 연다.
4. **Expose HTTP Ports (Max 10)** 에 `11434`를 추가한다. template에 `8888`이 있다면 지우지 않고 `8888, 11434`처럼 함께 둔다.
5. **Environment variables**에 key `OLLAMA_HOST`, value `0.0.0.0`을 추가한다.
6. **Set Overrides → Deploy On-Demand**를 선택해 Pod를 실행한다.

이미 실행 중인 Pod라면 Pod 메뉴의 **Edit Pod**에서 같은 port와 환경 변수를 추가한다. 변경 후 Pod가 다시 시작되면 Web Terminal에 재접속한다. `11434`는 Ollama의 내부 HTTP port이고, `0.0.0.0`은 Pod 외부에서 들어온 요청도 server가 받도록 하는 binding 주소이다.

## Web Terminal에서 Ollama 설치와 모델 준비

Pod 상태가 `Running`이 되면 **Connect → Enable Web Terminal → Open Web Terminal**로 접속한다. 다음 명령은 노트북 코드셀이 아니라 RunPod Web Terminal에서 실행한다.

먼저 공식 가이드의 의존성을 설치하고 Ollama server를 background로 시작한다. log는 `/workspace/ollama.log`에 남기므로 설치 또는 실행 오류를 다시 확인할 수 있다.

```bash
apt update && apt install -y lshw zstd
(curl -fsSL https://ollama.com/install.sh | sh && ollama serve > /workspace/ollama.log 2>&1) &
```

## Ollama server와 기본 모델 확인하기

server가 준비될 때까지 log를 확인한다. `Listening` 메시지가 나타나면 `Ctrl+C`로 log 보기만 종료하며 background의 Ollama server는 계속 실행된다.

```bash
tail -f /workspace/ollama.log
```

01번 노트북에서 내려받은 기본 모델이 남아 있는지 확인한다. `ollama list`와 `/api/tags`의 model 목록에 `llama3.2`가 보이면 외부 공개 실습을 시작할 수 있다.

```bash
ollama -v
ollama list
curl http://localhost:11434/api/tags
```

목록에 `llama3.2`가 없다면 01번 노트북의 `ollama pull llama3.2`를 먼저 실행한다.


## Pod 내부 주소와 외부 proxy 주소

하나의 Ollama 서버도 요청을 보내는 위치에 따라 주소가 달라진다. `localhost`는 현재 컴퓨터 자신을 뜻하므로 RunPod 터미널에서는 Pod를 가리키지만, 로컬 PyCharm에서는 학생 PC를 가리킨다.

- **Pod 내부 확인**: `http://localhost:11434`
- **외부 PyCharm 접속**: `https://OLLAMA_POD_ID-11434.proxy.runpod.net`

RunPod HTTP proxy는 내부 port `11434`를 HTTPS 주소로 전달한다. 따라서 외부 proxy URL 끝에 `:11434`를 다시 붙이지 않는다. 공식 URL 형식은 [RunPod port 공개 문서](https://docs.runpod.io/pods/configuration/expose-ports)의 `https://[POD_ID]-[INTERNAL_PORT].proxy.runpod.net`이다.

이름이 비슷한 server 설정과 client 주소의 역할도 다르다.

- `OLLAMA_HOST=0.0.0.0`: **서버가** 모든 network interface의 요청을 받도록 binding한다. RunPod의 Pod 환경 변수로 설정한다.
- `OLLAMA_BASE_URL=...`: **클라이언트가** 요청할 proxy root 주소를 담는 이 노트북의 Python 상수이다.

## Pod 내부에서 Ollama API 확인하기

RunPod Web Terminal에서 먼저 다음 명령을 실행한다. `/api/tags`는 서버에 내려받은 모델 목록을 반환하므로 Ollama server와 model 준비 상태를 함께 확인할 수 있다.

```bash
curl http://localhost:11434/api/tags
```

응답의 `models` 목록에서 `llama3.2`를 확인한다. 이어서 Pod 내부에서 짧은 생성 요청을 보내 model을 GPU memory에 먼저 적재한다. 이 준비 요청은 RunPod proxy의 100초 제한과 관계없이 cold load를 끝내기 위해 사용한다.

```bash
curl -X POST http://localhost:11434/api/generate   -H 'Content-Type: application/json'   -d '{
    "model": "llama3.2",
    "prompt": "RAG를 한 문장으로 설명해 줘.",
    "stream": false,
    "options": {"num_predict": 32}
  }'
```


http://localhost:11434 = find my pod Ollama /
api/generate = ask request



두 요청 중 하나라도 실패한다면 외부 proxy가 아니라 `ollama serve` 실행 상태와 model 다운로드를 먼저 확인한다.


## 외부 터미널에서 RunPod proxy 확인하기

이번 명령은 RunPod Web Terminal이 아니라 **로컬 PC의 PyCharm Terminal**에서 실행한다. `OLLAMA_POD_ID`를 현재 Pod 화면에 표시된 실제 ID로 바꾼다.

```bash
curl https://OLLAMA_POD_ID-11434.proxy.runpod.net/api/tags

curl https://01ai49n7okj3bz-11434.proxy.runpod.net/api/tags
```

Pod 내부 요청은 성공하지만 외부 요청이 `404`라면 오래된 Pod ID를 사용했거나 `11434`가 HTTP port로 공개되지 않은 경우가 많다. connection refused라면 Ollama server와 `OLLAMA_HOST=0.0.0.0`을 확인한다. 이 외부 `curl`이 정상 JSON을 반환한 뒤에만 Python client 실습으로 진행한다.

## native Ollama API로 generate와 chat 요청하기

Ollama native API는 `/api` 아래에 기능별 endpoint를 제공한다. `/api/generate`는 하나의 prompt를 받아 문자열을 생성하고, `/api/chat`은 `system`·`user`·`assistant` 역할이 있는 message 목록을 받는다.

다음 명령은 bash/zsh 형식이며 로컬 PC의 PyCharm Terminal에서 실행한다. Windows에서는 뒤의 Python client 실습으로 같은 endpoint를 확인할 수 있다. Ollama는 기본적으로 여러 JSON 조각을 streaming하므로, 첫 확인에서는 `stream: false`로 한 JSON 응답을 받는다. `num_predict`는 답변 길이를 제한해 수업용 요청이 RunPod proxy의 100초 제한에 걸릴 가능성을 줄인다.

```bash
curl -X POST https://OLLAMA_POD_ID-11434.proxy.runpod.net/api/generate   -H 'Content-Type: application/json'   -d '{
    "model": "llama3.2",
    "prompt": "RAG를 한 문장으로 설명해 줘.",
    "stream": false,
    "options": {"num_predict": 64}
  }

curl -X POST https://01ai49n7okj3bz-11434.proxy.runpod.net/api/generate   -H 'Content-Type: application/json'   -d '{
    "model": "llama3.2",
    "prompt": "RAG를 한 문장으로 설명해 줘.",
    "stream": false,
    "options": {"num_predict": 64}
  }'
```

정상 응답에서는 생성 문장이 `response`에 들어간다. 다음 요청은 같은 model에 역할이 있는 message 목록을 전달하며, 답변은 `message.content`에 들어간다.

```bash
curl -X POST https://OLLAMA_POD_ID-11434.proxy.runpod.net/api/chat   -H 'Content-Type: application/json'   -d '{
    "model": "llama3.2",
    "messages": [
      {"role": "user", "content": "왜 하늘은 파란색이야?"}
    ],
    "stream": false,
    "options": {"num_predict": 64}
  }'
```

요청 형식과 반환 필드는 [Ollama API 문서](https://docs.ollama.com/api/introduction)에서 확인할 수 있다.


## PyCharm Python 환경 준비하기

외부 접속에 사용할 공식 Ollama Python client와 LangChain 연동 package를 설치한다. 이 코드부터는 **RunPod 외부 Jupyter kernel이 아닌 로컬 PyCharm kernel**에서 실행한다. 수업 중 API 차이로 인한 오류를 줄이기 위해 확인한 두 package 버전을 함께 고정한다.

In [1]:
%pip install -U "ollama==0.6.2" "langchain-ollama==1.1.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]
Note: you may need to restart the kernel to use updated packages.


## 원격 주소와 model tag 설정하기

`POD_ID`에 RunPod 화면의 실제 Pod ID를 입력한다. 이 값으로 proxy root 주소를 만들고, `OLLAMA_MODEL`에는 01번 노트북에서 내려받은 model tag `llama3.2`를 지정한다. 주소에는 `/api/chat`이나 `/api/generate`를 붙이지 않으며 각 client가 endpoint path를 자동으로 추가한다.

Pod를 새로 만들면 Pod ID가 바뀌므로 이 셀의 값도 다시 입력한다. URL과 model tag는 비밀값이 아니지만 공개 endpoint에는 민감한 prompt를 보내지 않는다.


In [2]:
POD_ID = 'OLLAMA_POD_ID'

OLLAMA_BASE_URL = f'https://wcgzlvuig08tw5-11434.proxy.runpod.net'
OLLAMA_MODEL = 'llama3.2'


## 공식 Ollama Client로 model 목록 확인하기

`Client(host=...)`는 지정한 root 주소 뒤에 native `/api` endpoint를 붙이는 공식 Python client이다. 먼저 `list()`로 `/api/tags`와 같은 model 목록을 받아 `OLLAMA_MODEL`의 tag가 실제 서버에 있는지 확인한다.

In [3]:
from ollama import Client

ollama_client = Client(host=OLLAMA_BASE_URL)

model_list = ollama_client.list()

print(model_list)

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 8, 26, 5, 14, 4, 268809, tzinfo=TzInfo(0)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


## 공식 Ollama Client로 chat 요청하기

`chat()`은 역할이 있는 message 목록을 native `/api/chat`에 전달한다. 입력 message는 `role`과 `content`를 가지며, 생성 결과는 응답의 `message.content`에 들어간다.

In [7]:
ollama_response = ollama_client.chat(
    model = OLLAMA_MODEL,
    messages= [
        {
            "role": "user",
            "content": '혈당 스파이크 예방하는 방법 알려줘.'
        }
    ],
    stream=False,
)
print(ollama_response.message.content)

혈당 스파이크 예방하는 방법은 다음과 같습니다.

1. **적절한 आ료**: 적절한 amount의 carbohydrate와 protein을 섭취하며, 과립당화의 적절한 수준을 유지하는 것이 중요합니다. 과립당화가 적절하게 관리되지 않으면 insulin resistance가 발생할 수 있습니다.
2. **산소의 충분한 섭취**: 산소의 충분한 섭취는 혈당 스파이크 예방에 도움이 됩니다. 산소는 혈액을 수용한 후 glucose를 분해하는 데 도움을 줍니다.
3. **신선한อาหาร**: 신선한 alimentos을 섭취하는 것이 혈당 스파이크 예방에 도움이 됩니다. 신선한 alimentos은 비타민, 미네랄, 아미노산, 및 andere 성분이 풍부합니다.
4. **중성 피로**: 중성 피로(유기적 에너지)가 충분히 유지되는 것이 혈당 스파이크 예방에 도움이 됩니다. 중성 피로가 충분하지 않으면 혈당이 spikes할 수 있습니다.
5. **수분 보충**: 수분 보충이 중요합니다. 수분 보충이不足하면 혈당 스파이크 예방에 도움이 없습니다.
6. **진로**: 진로는 혈당 스파이크 예방에 도움이 됩니다. 진로가 충분하지 않으면 혈당이 spikes할 수 있습니다.
7. **스트레스 관리**: 스트레스 관리는 혈당 스파이크 예방에 도움이 됩니다. 스트레스는 혈당을 높일 수 있습니다.
8. **기상 조절**: 기상 조절은 혈당 스파이크 예방에 도움이 됩니다. 기상 조절이 충분하지 않으면 혈당이 spikes할 수 있습니다.
9. **유기적 에너지**: 유기적 에너지(비중성 피로)가 충분히 유지되는 것이 혈당 스파이크 예방에 도움이 됩니다.
10. **진료**: 혈당 스파이크 예방을 위해 진료가 필요할 수 있습니다. 혈당 스파이크를 예방하는 방법은 다양한 방법이 있으므로, 의사와의 협력을 통해 적절한 예방 방법을 찾으세요.

위의 방법은 혈당 스파이크 예방에 도움이 될 수 있지만, 항상 의사와의 협력을 통해 적절한 예방 방법을 찾으세요.


## ChatOllama로 LangChain에 연결하기

`ChatOllama`는 같은 native Ollama API를 LangChain의 ChatModel 인터페이스로 감싼다. `invoke()`가 반환하는 `AIMessage`는 prompt·chain·agent 같은 LangChain 구성 요소에 바로 연결할 수 있다.

`base_url`에는 proxy root 주소를 전달한다. `server_url`과 `server_type`은 현재 `ChatOllama` 생성자 인자가 아니므로 사용하지 않는다.

In [9]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv(override=False)

remote_model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.3,
    num_predict=128, # 최대 Token 수
)

langchain_response = remote_model.invoke('RAG 장점 두가지 설명.')
print(langchain_response.content)
print(langchain_response.response_metadata.get('model'))

RAG (Reactive Attachment Disorder)는 유아기에서 부모-아동의 부드러운 결합을 보지 못하는 경우를指します. RAG의 장점은 다음과 같습니다.

1. **자율성**: RAG의 아동은 자율성을 더 잘 발휘할 수 있습니다. 부모-아동의 결합이 부드럽지 못하면, 아동은 자율성을 발휘하는 것을 더 잘 할 수 있습니다. 이는 아동이 자신의 의도와 원리를 따라 행동하는 것을 유도합니다.
2. **자아존중감**:
llama3.2


## 다음 실습에서 News2Stock 모델 재사용하기

04번 노트북에서 News2Stock GGUF를 Ollama model tag `news2stock`으로 등록한 뒤에는 이 노트북의 주소와 호출 코드를 그대로 재사용할 수 있다. `OLLAMA_MODEL` 값만 `news2stock`으로 바꾸면 `/api/generate`, `/api/chat`, 공식 Ollama Client와 `ChatOllama`가 같은 custom model을 호출한다.